# Sentinel-1 burst SLCs matched to NISAR acquisition dates (Alaska)

For the same ISMN stations used in the Alaska NISAR GSLC pipeline
(`Earsel26_NISAR-main/notebooks/ismn_data/alaska/stations.geojson`), this notebook:

1. Loads each station's NISAR acquisition dates from the saved GSLC window files
   (`nisar_data/alaska_GSLC_windows/`) -- unlike the Spain AOI, a station here is
   typically covered by **several different NISAR tracks**, so this is a per-station
   set of dates, not one shared stack-wide list.
2. Searches ASF's catalog for Sentinel-1 burst SLC products in a small (80x80m)
   box around each station, **keeping every relative orbit that covers it**, not
   just one dominant orbit -- Alaska stations are covered by several distinct S1
   passes (ascending/descending, different paths), the same way they're covered by
   several distinct NISAR tracks.
3. Per (station, orbit), greedily matches the closest available Sentinel-1 date to
   each of that station's NISAR dates, up to the same count -- so each kept orbit's
   sample size matches the NISAR sample size it's being compared against.
4. **Discards any (station, orbit) combination with fewer than 3 matched dates** --
   the De Zan phase-closure inversion needs at least a triplet, so an orbit that
   can't supply one isn't useful here.
5. Downloads the selected raw burst SLCs (not HyP3 on-demand InSAR-pair products),
   so the same local coherence / phase-closure computation used elsewhere in this
   project can later be applied to Sentinel-1 too.

Requires an Earthdata login (same credentials as `earthaccess`/HyP3) and
`asf_search`, `shapely`, `pandas`, `numpy`.

:::{note} Requires the Alaska GSLC windows to already be downloaded
Target dates come from `Earsel26_NISAR-main/.../nisar_data/alaska_GSLC_windows/*.nc`
(via `notebooks/03_get_NISAR_GSLC_windows.ipynb`, despite the filename --
Alaska is the current AOI). If that directory is empty, every station will show 0
target dates.
:::

In [ ]:
import sys
import os
from pathlib import Path

# Same PROJ database fix as NISAR_SM.ipynb / Checks.ipynb -- this env's own
# bundled pyproj data is broken (CRSError: "no database context specified").
import pyproj
pyproj.datadir.set_data_dir(os.path.join(sys.prefix, "Library", "share", "proj"))


def require_globals(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Missing {missing} -- run the earlier setup cells first "
            "(the kernel may have been restarted)."
        )


BASE = Path.cwd()

# The Alaska GSLC pipeline (station list, saved windows, and the load_station_
# acquisitions loader) lives in the Earsel26_NISAR-main repo, not here.
EARSEL_ROOT = BASE / "Earsel26_NISAR-main"
EARSEL_NOTEBOOKS = EARSEL_ROOT / "notebooks"
if str(EARSEL_ROOT) not in sys.path:
    sys.path.append(str(EARSEL_ROOT))


## 1. Station list and per-station NISAR acquisition dates

Reuses `load_station_acquisitions` from
[`Earsel26_NISAR-main/util/nisar_coherence_closure.py`](../Earsel26_NISAR-main/util/nisar_coherence_closure.py)
-- the same loader notebooks 04 and 05 in that repo use -- rather than re-deriving
dates from the raw `.nc` filenames here. A station's dates are the union across
*all* of its kept NISAR tracks: NISAR's own same-track requirement is handled
independently over on that side (each NISAR track already only gets compared
against SAR data from the same track), so this side only needs "which calendar
dates does an estimate exist for at this station", not "for which NISAR track".

In [ ]:
require_globals("EARSEL_NOTEBOOKS")

import json

from util.nisar_coherence_closure import load_station_acquisitions

stations_path = EARSEL_NOTEBOOKS / "ismn_data" / "alaska" / "stations.geojson"
with open(stations_path) as f:
    stations_geojson = json.load(f)

stations = {
    feat["properties"]["station_val"]: (
        feat["geometry"]["coordinates"][1],  # lat
        feat["geometry"]["coordinates"][0],  # lon
    )
    for feat in stations_geojson["features"]
}

gslc_dir = EARSEL_NOTEBOOKS / "nisar_data" / "alaska_GSLC_windows"

nisar_dates_by_station = {}
for name in stations:
    groups = load_station_acquisitions(gslc_dir, name)
    dates = sorted({acq["datetime"] for track_acqs in groups.values() for acq in track_acqs})
    nisar_dates_by_station[name] = dates

print(f"{len(stations)} stations")
for name, dates in nisar_dates_by_station.items():
    print(f"  {name}: {len(dates)} NISAR acquisition date(s)")


## 2. Earthdata / ASF login

In [ ]:
from getpass import getpass
import asf_search as asf

_username = input("Earthdata username: ")
_password = getpass("Earthdata password: ")
asf_session = asf.ASFSession().auth_with_creds(_username, _password)
del _password


## 3. Per-station Sentinel-1 burst search, all covering orbits

One search per station (not one combined search across all stations, unlike the
Spain-AOI version of this notebook) -- Alaska's stations are spread across the
whole state, hundreds of km apart, so a single shared bounding box would be huge
and wouldn't correspond to any one consistent geometry anyway. Each station's
search box is the station's own **80x80m footprint** (matching the NISAR GSLC /
HLS window size used elsewhere in this project), buffered from a point to a box
so a burst has to fully *contain* that footprint, not just intersect the point,
to count as covering the station (Section 4 below).

In [ ]:
require_globals("stations", "nisar_dates_by_station", "asf_session")

import numpy as np
import pandas as pd
from dateutil.parser import parse as parse_date
from shapely.geometry import shape, box as shapely_box

WINDOW_FOOTPRINT_M = 80.0  # matches the NISAR GSLC / HLS window
_HALF_M = WINDOW_FOOTPRINT_M / 2
PAD_DAYS = pd.Timedelta(days=15)  # past the NISAR date range, so edge dates still have real candidates on both sides

METERS_PER_DEG_LAT = 111_320.0


def _station_box_deg(lat, lon, half_m=_HALF_M):
    """(minlon, minlat, maxlon, maxlat) of a half_m-radius box around (lat, lon),
    in degrees -- longitude spacing shrinks with cos(latitude), which matters at
    Alaska's high latitudes (60-70N) far more than it would near the equator."""
    dlat = half_m / METERS_PER_DEG_LAT
    dlon = half_m / (METERS_PER_DEG_LAT * np.cos(np.radians(lat)))
    return lon - dlon, lat - dlat, lon + dlon, lat + dlat


station_search_results = {}
for name, (lat, lon) in stations.items():
    nisar_dates = nisar_dates_by_station[name]
    if not nisar_dates:
        print(f"{name}: no NISAR dates, skipping search")
        station_search_results[name] = []
        continue

    minlon, minlat, maxlon, maxlat = _station_box_deg(lat, lon)
    search_polygon = (
        f"POLYGON(({minlon} {minlat},{maxlon} {minlat},"
        f"{maxlon} {maxlat},{minlon} {maxlat},{minlon} {minlat}))"
    )
    search_start = (min(nisar_dates) - PAD_DAYS).to_pydatetime()
    search_end = (max(nisar_dates) + PAD_DAYS).to_pydatetime()

    results = asf.search(
        platform=asf.PLATFORM.SENTINEL1,
        polarization=asf.POLARIZATION.VV,
        intersectsWith=search_polygon,
        start=search_start,
        end=search_end,
        processingLevel=asf.PRODUCT_TYPE.BURST,
        beamMode=asf.BEAMMODE.IW,
    )
    station_search_results[name] = results
    orbits = sorted({r.properties["pathNumber"] for r in results})
    print(f"{name}: {len(results)} burst product(s) across {len(orbits)} relative orbit(s) {orbits}")


## 4. Per (station, orbit): covering burst, date matching, triplet discard

For each station, each relative orbit found above:

1. Find the burst ID whose footprint **fully contains** the station's 80x80m box
   (not just its center point) -- one footprint check per unique burst ID, since
   the same burst ID's footprint repeats every pass.
2. Greedily match that burst's available dates to the station's NISAR dates (same
   global-nearest-first matching as before), up to the NISAR date count.
3. **Discard this (station, orbit) if fewer than 3 dates matched** -- printed, not
   silently dropped, so a station losing an orbit here is visible.

In [ ]:
require_globals("station_search_results", "stations", "nisar_dates_by_station")

MIN_TRIPLET = 3  # De Zan phase closure needs at least 3 acquisitions


def _parse_s1_date(start_time_str):
    # ASF's startTime is an ISO8601 string with an explicit UTC ("Z") suffix, which
    # parses tz-aware -- but nisar_dates (from acquisition_datetime_utc, no tz
    # marker in NISAR's own zeroDopplerStartTime string) parses tz-naive, and
    # match_nearest_dates subtracts one from the other. Both already represent UTC
    # wall-clock time, so stripping tz here (rather than adding one to the NISAR
    # side) keeps every date in this notebook consistently tz-naive.
    ts = pd.Timestamp(parse_date(start_time_str)).normalize()
    return ts.tz_localize(None) if ts.tzinfo is not None else ts


def match_nearest_dates(candidate_dates, target_dates):
    """Greedy global-nearest matching: among all (target, candidate) pairs,
    repeatedly take the closest unused pair until every target is matched or
    candidates run out. Returns {target_date: candidate_date}."""
    pairs = sorted(
        ((abs((t - c).total_seconds()), t, c) for t in target_dates for c in candidate_dates),
        key=lambda x: x[0],
    )
    matched = {}
    used_candidates = set()
    for _, t, c in pairs:
        if t in matched or c in used_candidates:
            continue
        matched[t] = c
        used_candidates.add(c)
        if len(matched) == len(target_dates):
            break
    return matched


station_s1_selection = {}  # station -> orbit -> {nisar_date: (s1_date, result)}
for name, (lat, lon) in stations.items():
    results = station_search_results.get(name, [])
    nisar_dates = nisar_dates_by_station[name]
    if not results or not nisar_dates:
        continue

    station_box = shapely_box(*_station_box_deg(lat, lon))

    orbit_results = {}
    for r in results:
        orbit_results.setdefault(r.properties["pathNumber"], []).append(r)

    kept_orbits = {}
    for orbit, orbit_products in sorted(orbit_results.items()):
        burst_footprints = {}
        for r in orbit_products:
            burst_id = r.properties["burst"]["fullBurstID"]
            if burst_id not in burst_footprints:
                burst_footprints[burst_id] = shape(r.geometry)

        covering_burst_id = next(
            (bid for bid, poly in burst_footprints.items() if poly.contains(station_box)), None
        )
        if covering_burst_id is None:
            print(f"  {name} orbit {orbit}: no single burst fully covers the 80x80m footprint, skipping")
            continue

        by_date = {}
        for r in orbit_products:
            if r.properties["burst"]["fullBurstID"] != covering_burst_id:
                continue
            d = _parse_s1_date(r.properties["startTime"])
            by_date.setdefault(d, r)

        matched = match_nearest_dates(list(by_date.keys()), nisar_dates)
        if len(matched) < MIN_TRIPLET:
            print(f"  {name} orbit {orbit}: only {len(matched)}/{len(nisar_dates)} dates matched "
                  f"(< {MIN_TRIPLET}), discarding -- no triplet possible")
            continue

        kept_orbits[orbit] = {nisar_d: (s1_d, by_date[s1_d]) for nisar_d, s1_d in matched.items()}
        print(f"  {name} orbit {orbit}: {len(matched)}/{len(nisar_dates)} dates matched, kept")

    if kept_orbits:
        station_s1_selection[name] = kept_orbits
    else:
        print(f"{name}: no orbit had a usable triplet, station dropped entirely")

n_kept_stations = len(station_s1_selection)
n_kept_orbits = sum(len(o) for o in station_s1_selection.values())
print(f"\n{n_kept_stations}/{len(stations)} stations have at least one usable orbit "
      f"({n_kept_orbits} station-orbit combination(s) total)")


## 5. Download selected burst SLCs

Deduplicated across every (station, orbit) combination first -- neighboring
stations sharing a burst, or two orbits partially overlapping the same date's
product, would otherwise download the same file twice.

In [ ]:
require_globals("station_s1_selection", "BASE")

import json
import time

import requests

# The CloudFront/S3 distribution behind these presigned URLs occasionally drops
# the connection mid-download under load (RemoteDisconnected/ConnectionError) --
# result.download()'s own retry only covers "burst still processing" responses,
# not this, so it otherwise propagates immediately and kills the whole loop.
# Same retry-with-backoff pattern used for transient errors in
# Earsel26_NISAR-main/util/nisar_station_timeseries.py.
MAX_DOWNLOAD_RETRIES = 5
DOWNLOAD_RETRY_BACKOFF_SECONDS = 5


def _download_with_retry(result, path, session, file_type):
    for attempt in range(MAX_DOWNLOAD_RETRIES):
        try:
            result.download(path=path, session=session, fileType=file_type)
            return
        except requests.exceptions.RequestException as e:
            if attempt == MAX_DOWNLOAD_RETRIES - 1:
                raise
            wait_s = DOWNLOAD_RETRY_BACKOFF_SECONDS * (2 ** attempt)
            print(f"    transient {type(e).__name__}, retrying in {wait_s}s "
                  f"(attempt {attempt + 1}/{MAX_DOWNLOAD_RETRIES})")
            time.sleep(wait_s)


s1_dir = BASE / "S1_burst_data_alaska"
s1_dir.mkdir(exist_ok=True)

unique_products = {}
for orbits in station_s1_selection.values():
    for matches in orbits.values():
        for s1_d, result in matches.values():
            unique_products[result.properties["fileID"]] = result

print(f"Downloading {len(unique_products)} unique burst SLCs to {s1_dir} ...")
for file_id, result in unique_products.items():
    # A burst product is a .tiff (raster) plus a .xml (annotation/calibration
    # metadata, needed later to geocode the burst) -- ALL_FILES fetches both,
    # named from properties['fileName'] rather than fileID (they can differ).
    base_filename = ".".join(result.properties["fileName"].split(".")[:-1])
    out_path = s1_dir / f"{base_filename}.tiff"
    if out_path.exists():
        print(f"  {file_id}: already downloaded, skipping")
        continue
    _download_with_retry(result, str(s1_dir), asf_session, asf.FileDownloadType.ALL_FILES)
    print(f"  {file_id}: downloaded")

# Manifest: station -> orbit -> matched (nisar_date, s1_date, file_id) triples, so
# a later coherence-computation pass doesn't need to redo the search/matching.
manifest = {}
for name, orbits in station_s1_selection.items():
    manifest[name] = {
        str(orbit): [
            {
                "nisar_date": nisar_d.isoformat(),
                "s1_date": s1_d.isoformat(),
                "file_id": result.properties["fileID"],
            }
            for nisar_d, (s1_d, result) in sorted(matches.items())
        ]
        for orbit, matches in orbits.items()
    }

manifest_path = s1_dir / "station_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"\nWrote manifest for {len(manifest)} station(s) to {manifest_path}")


## Next step (not yet implemented)

`S1_burst_data_alaska/station_manifest.json` maps each station and each kept
relative orbit to its matched NISAR/Sentinel-1 date pairs and downloaded burst
SLC files. Turning these burst SLCs into an actual coherence / phase-closure
soil-moisture estimate still needs: extracting each burst's complex SLC pixels
at the station location (a burst has its own per-burst geocoding, unlike NISAR's
already-geocoded GSLC grid), and running the same `multilook_coherence` /
`calc_SM_coherences` / `invert_sm` pipeline on the resulting per-(station, orbit)
window stack -- independently per orbit, the same way NISAR's own per-track
groups are each inverted independently.